# Convert JSON to PDF
This notebook reads `data/consumer_complaints.json`, formats the raw text data with word wrapping, and exports it directly to `data/consumer_complaints.pdf`.

In [1]:
!pip install reportlab

In [ ]:
import json
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.colors import HexColor

INPUT_JSON = 'data/consumer_complaints.json'
OUTPUT_PDF = 'data/consumer_complaints.pdf'

print('Loading JSON...')
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Using len(data) to process EVERY record the file holds
limit = len(data)
print(f'Starting PDF generation for all {limit} records...')

doc = SimpleDocTemplate(OUTPUT_PDF, pagesize=letter)
styles = getSampleStyleSheet()
title_style = styles['Heading3']
title_style.textColor = HexColor('#003366')
meta_style = styles['Italic']
meta_style.textColor = HexColor('#555555')
body_style = styles['BodyText']
body_style.spaceAfter = 14

Story = [Paragraph('<b>Consumer Complaints Document Export</b>', styles['Heading1']), Spacer(1, 12)]

for i, item in enumerate(data[:limit], 1):
    struct = item.get('StructuredData', {})
    headline = item.get('Title/Headline', 'No Title')
    cat = struct.get('Fraud Category', 'Unknown')
    subcat = struct.get('Fraud Subcategory', 'Unknown')
    date = struct.get('Original Date', 'Unknown')
    f_text = item.get('Full Text', '')
    
    # Sanitize texts so standard PDF drawer doesn't crash on unrecognized utf-8 symbols/emojis
    headline = headline.replace('<', '&lt;').replace('>', '&gt;').replace('\n', ' ')
    headline = headline.encode('latin-1', 'replace').decode('latin-1')
    
    f_text = f_text.replace('<', '&lt;').replace('>', '&gt;').replace('\n', '<br/>')
    f_text = f_text.encode('latin-1', 'replace').decode('latin-1')
    
    if len(f_text) > 1000:
        f_text = f_text[:1000] + '... [TRUNCATED]'
        
    Story.append(Paragraph(f'{i}. {headline}', title_style))
    nav = f"<b>Category:</b> {cat} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Subcat:</b> {subcat} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Date:</b> {date}"
    Story.append(Paragraph(nav, meta_style))
    Story.append(Spacer(1, 6))
    Story.append(Paragraph(f_text, body_style))

print(f'Building PDF at {OUTPUT_PDF}...')
doc.build(Story)
print(f'✅ PDF generated successfully into {OUTPUT_PDF}!')

Loading JSON...
Starting PDF generation for the top 500 records...
Building PDF at data/consumer_complaints.pdf...
✅ PDF generated successfully into data/consumer_complaints.pdf!
